# yt-clips — One-Shot Colab Worker
1. Runtime → T4 GPU
2. Add `OPENROUTER_API_KEY` to Colab Secrets (🔑 left panel)
3. Run the cell below — that's it

In [ ]:
import os, subprocess, sys, time
from pathlib import Path

def sh(cmd, timeout=120):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    if r.returncode and r.stderr: print(r.stderr.strip()[-200:])
    return r

print("="*55)
print("  ONE-SHOT SETUP")
print("="*55)

# ── Mount Drive & chdir ──
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
for p in ["/content/drive/MyDrive/yt-clips", "/content/drive/My Drive/yt-clips"]:
    if Path(p).exists(): os.chdir(p); break
print(f"PWD: {os.getcwd()}")

# ── Force-pull latest ──
if Path(".git").exists():
    sh("git stash 2>/dev/null", 10)
    sh("git fetch origin main", 30)
    sh("git reset --hard origin/main", 15)
else:
    sh("git clone https://github.com/fearless16/yt-clips.git /tmp/yt-clips", 60)
    os.chdir("/tmp/yt-clips")
sys.path.insert(0, ".")
assert Path("automation/__init__.py").exists(), "automation/ missing after pull"

# ── Load secrets ──
for line in open(".env") if Path(".env").exists() else []:
    line = line.strip()
    if line and "=" in line and not line.startswith("#"):
        k, v = line.split("=", 1); os.environ[k.strip()] = v.strip()
try:
    from google.colab import userdata
    for k in ["OPENROUTER_API_KEY", "GROQ_API_KEY", "GOOGLE_API_KEY"]:
        v = userdata.get(k)
        if v: os.environ[k] = v
except: pass

# ── Install deps ──
print("Installing system deps...")
sh("apt-get update -qq && apt-get install -y -qq aria2 ffmpeg >/dev/null 2>&1")
print("Installing torch+cu121...")
sh("pip install -q torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu121", 300)
print("Installing pipeline deps...")
sh("pip install -q yt-dlp faster-whisper rich PyYAML opencv-python-headless numpy "
   "filterpy scipy openai python-dotenv Pillow requests "
   "ultralytics gfpgan basicsr realesrgan "
   "google-api-python-client google-auth-httplib2 google-auth-oauthlib")

# ── Verify ──
import automation; print(f"automation v{automation.VERSION}")
import torch
print(f"CUDA: {torch.cuda.is_available()} | {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU'}")

# ── Dirs + cleanup ──
for d in ["input","temp","transcripts","highlights","shorts","logs","photos"]:
    Path(d).mkdir(exist_ok=True)
sh("pkill -f 'python watcher.py' 2>/dev/null; pkill -f serveo 2>/dev/null; fuser -k 5000/tcp 2>/dev/null; sleep 2")

# ── Start watcher ──
sh(f"nohup {sys.executable} watcher.py > watcher.log 2>&1 &")
time.sleep(4)
ok = subprocess.run("curl -sf http://localhost:5000/health", shell=True, capture_output=True).returncode == 0
print(f"Watcher: {'OK' if ok else 'FAILED'}")
if not ok: print(open("watcher.log").read()[-300:])

# ── Start tunnel ──
def tunnel(cmd, wait=8):
    sh(cmd.format(L="tunnel.log"), 15); time.sleep(wait)
    if Path("tunnel.log").exists():
        for line in open("tunnel.log"):
            for w in line.split():
                w = w.strip().rstrip(",.;")
                if w.startswith("https://"):
                    Path("colab_url.txt").write_text(w); return w
    return None
url = tunnel("nohup ssh -o StrictHostKeyChecking=no -o ServerAliveInterval=30 -R 80:localhost:5000 serveo.net > {L} 2>&1 &")
if not url:
    print("serveo failed → localhost.run...")
    sh("pkill -f serveo 2>/dev/null; sleep 2")
    url = tunnel("nohup ssh -o StrictHostKeyChecking=no -o ServerAliveInterval=30 -R 80:localhost:5000 nokey@localhost.run > {L} 2>&1 &", 12)
print(f"Tunnel: {url or 'N/A'}")

print("\n" + "="*55)
print("  WORKER ONLINE — live updates every 15s")
print("  Send jobs: python bridge.py \"https://youtu.be/...\"")
print("="*55 + "\n")

# ── Dashboard loop (keeps cell alive) ──
pos = 0
while True:
    time.sleep(15)
    log = Path("watcher.log")
    if log.exists():
        with open(log) as f:
            f.seek(pos)
            for l in f:
                if l.strip(): print(l.strip())
            pos = f.tell()
    mem = Path("/proc/meminfo")
    free = int(open(mem).read().split("MemFree:")[1].split()[0])/1e6 if mem.exists() else 0
    has_gpu = torch.cuda.is_available()
    gpu = subprocess.run(["nvidia-smi","--query-gpu=name,memory.free","--format=csv,nounits,noheader"],
                        capture_output=True,text=True,timeout=5).stdout.strip() if has_gpu else "?"
    print(f"[RAM {free:.1f}GB | GPU {gpu[:30]} | Tunnel {'OK' if Path('colab_url.txt').exists() else '...'}]")

